In [ ]:
# Import required libraries

import numpy as np
import pandas as pd

In [ ]:
# Feature Engineering Function

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies astronomical feature engineering to the SDSS dataset.
    Returns a new DataFrame with enhanced features for ML models.
    """
    df = df.copy()
    
    # ---------------------------------------------------------
    # 1. Color Indices (Spectral Energy Distribution Proxies)
    # ---------------------------------------------------------
    # Differences between adjacent bands indicate the temperature/composition.
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    
    # Non-adjacent color indices for broader spectral slope analysis
    df['u_r'] = df['u'] - df['r']
    df['u_z'] = df['u'] - df['z']
    df['g_z'] = df['g'] - df['z']
    
    # ---------------------------------------------------------
    # 2. Photometric Approximations
    # ---------------------------------------------------------
    # Average magnitude across all bands
    df['avg_mag'] = (df['u'] + df['g'] + df['r'] + df['i'] + df['z']) / 5.0
    
    # ---------------------------------------------------------
    # 3. Spatial Coordinate Transformations
    # ---------------------------------------------------------
    # alpha (RA) and delta (Dec) are spherical coordinates in degrees.
    # RA wraps around at 360 -> 0. Tree models struggle with this wrapping.
    # Converting to 3D Cartesian coordinates solves the discontinuity.
    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])
    
    df['cartesian_x'] = np.cos(alpha_rad) * np.cos(delta_rad)
    df['cartesian_y'] = np.sin(alpha_rad) * np.cos(delta_rad)
    df['cartesian_z'] = np.sin(delta_rad)
    
    # ---------------------------------------------------------
    # 4. Redshift Interactions
    # ---------------------------------------------------------
    # Stars are in our galaxy, so their cosmological redshift is ~0.
    df['is_near_zero_redshift'] = (np.abs(df['redshift']) < 1e-4).astype(int)
    
    # Pseudo-Luminosity: Observed magnitude depends on distance. 
    # Combining magnitude with log(redshift) acts as a proxy for Absolute Brightness.
    safe_redshift = np.clip(df['redshift'], 0, None)
    df['pseudo_luminosity_r'] = df['r'] - 5 * np.log10(safe_redshift + 1e-5)
    
    # Pseudo K-Correction: Normalizing colors by (1 + z) to approximate rest-frame colors
    df['u_g_redshifted'] = df['u_g'] / (1 + safe_redshift)
    df['g_r_redshifted'] = df['g_r'] / (1 + safe_redshift)
    
    # ---------------------------------------------------------
    # 5. Linear Flux Transformations
    # ---------------------------------------------------------
    # Magnitudes are logarithmic. Converting back to relative linear flux
    # allows tree models to split on actual energy intensities.
    for col in ['u', 'g', 'r', 'i', 'z']:
        df[f'flux_{col}'] = 10 ** (-0.4 * df[col])
        
    # Total flux proxy
    df['total_flux'] = df['flux_u'] + df['flux_g'] + df['flux_r'] + df['flux_i'] + df['flux_z']
        
    return df

In [ ]:
if __name__ == "__main__":
    dataset_path = [
        "/kaggle/input/competitions/playground-series-s6e6/train.csv",
        "/kaggle/input/competitions/playground-series-s6e6/test.csv"
    ]
    for dataset in dataset_path:
        print("Loading data...")
        data = pd.read_csv(dataset)
        
        print(f"Original shape: {data.shape}")
        
        print("Engineering features...")
        data_engineered = engineer_features(data)
        
        print(f"Engineered shape: {data_engineered.shape}")
        print("\nNew features added:")
        new_cols = set(data_engineered.columns) - set(data.columns)
        for col in sorted(new_cols):
            print(f" - {col}")

        # Save the processed dataset in the output.
        file_name = "train" if "train.csv" in dataset else "test"
        data_engineered.to_parquet(f"processed_{file_name}.parquet",index=None)

        print(f"\n{file_name}.csv processed and saved.\n")
    
    print("\nDone! Data is ready for ML models.")
